# Adding Storages to the Energy System Model

In the [previous notebook](../_01_initialize/_1_initialize_ESM.ipynb), we initialized an energy system model, which defines the basic structure of the energy system such as locations, commodities, and the temporal resolution.

In this notebook, we introduce **Storages components**. Storages represent components that **can store a commodity and thus transfers it between time steps.**. We focus on the most essential parameters required to define and understand a Storage component, while more advanced and optional settings will be explained in subsequent notebooks.

Typical examples of storages include:

- Li-ion batteries to store electricity
- Depleted gas fields to store CO2
- Hot water tanks to store heat ...



## Add Storages

## Load ESM

We first load the ESM from the [previous notebook](../_02_add_component/_2_add_sink.ipynb).

In [1]:
import fine as fn
import fine.IOManagement.xarrayIO as xrIO
import pandas as pd
import numpy as np
from pathlib import Path
cwd = Path.cwd().resolve()
base_path = cwd.parents[2]
nc_file = base_path / "examples" / "Examples" / "NetCDF" / "esm_source_sink_transmission_conversion.nc"

esM = xrIO.readNetCDFtoEnergySystemModel(nc_file)

### Lithium Ion Batteries

The self discharge of a lithium ion battery is here described as 3% per month. The self discharge per hours is obtained using the equation (1-$\text{selfDischarge}_\text{hour})^{30*24\text{h}} = 1-\text{selfDischarge}_\text{month}$.

We can now add Lithium Ion batteries as a storage. Below you can find a more detailed explanation of the parameters used here.

In [2]:
esM.add(
    fn.Storage(
        esM=esM,
        name="Li-ion batteries",
        commodity="electricity",
        hasCapacityVariable=True,
        chargeEfficiency=0.95,
        cyclicLifetime=10000,
        dischargeEfficiency=0.95,
        selfDischarge=1 - (1 - 0.03) ** (1 / (30 * 24)),
        chargeRate=1,
        dischargeRate=1,
        doPreciseTsaModeling=False,
        investPerCapacity=0.151,
        opexPerCapacity=0.002,
        interestRate=0.08,
        economicLifetime=22,
    )
)

## Save the Energy System Model

In [3]:
cwd = Path.cwd().resolve()
base_path = cwd.parents[2]
nc_file = base_path / "examples" / "Examples" / "NetCDF" / "basic_esm.nc"

xrIO.writeEnergySystemModelToNetCDF(
    esM, outputFilePath=nc_file, overwriteExisting=True
)


Writing output to netCDF... 
Done. (0.6632 sec)


## General Structure of a Storage Instance

The following code snippet shows the extensive structure of the `Storage` class and its arguments. Since the structure contains many parameters with varying levels of relevance, it is for clarity indicated next to each parameter whether it is defined in this notebook or not.

```python
Storage(
    esM,                  
    name,                  
    commodity,          
    chargeRate=1,          
    dischargeRate=1,       
    chargeEfficiency=1,     
    dischargeEfficiency=1,  
    selfDischarge=0,
    cyclicLifetime=None,
    stateOfChargeMin=0,
    stateOfChargeMax=1,
    hasCapacityVariable=True,
    capacityVariableDomain="continuous",
    capacityPerPlantUnit=1,
    hasIsBuiltBinaryVariable=False,
    bigM=None,
    doPreciseTsaModeling=False,
    chargeOpRateMax=None,   
    chargeOpRateFix=None,   
    chargeTsaWeight=1,
    dischargeOpRateMax=None,
    dischargeOpRateFix=None,
    dischargeTsaWeight=1,
    isPeriodicalStorage=False,
    locationalEligibility=None,
    capacityMin=None,
    capacityMax=None,
    partLoadMin=None,
    sharedPotentialID=None,
    linkedQuantityID=None,
    capacityFix=None,
    commissioningMin=None,
    commissioningMax=None,
    commissioningFix=None,
    isBuiltFix=None,
    investPerCapacity=0,
    investIfBuilt=0,
    opexPerChargeOperation=0,
    opexPerDischargeOperation=0,
    opexPerCapacity=0,
    opexIfBuilt=0,
    interestRate=0.08,
    economicLifetime=10,
    technicalLifetime=None,
    floorTechnicalLifetime=True,
    socOffsetDown=-1,
    socOffsetUp=-1,
    stockCommissioning=None,
    pwlcfParameters=None,
)
```
In the following sections, we explain the most important arguments of a Storage component.


## Required Arguments

### esM

`esM` is the energy system model to which the storage is added.

### name

`name` is a string, which should describe the type of storage which is added to the energy system model.

Examples:
- "Li-ion_battery"
- "salt_cavern"

### commodity

`commodity` defines which commodity the storage should store.

The commodity must be one of the commodities that were defined when initializing the `EnergySystemModel`.

Examples of commodities that could be stored include:<br>
- electricity from renewable generation
- hydrogen from external supply
- heat

### hasCapacityVariable

`hasCapacityVariable` is a **boolean**, which specifies whether the component has a capacity limit.

Examples:<br>
- A wind turbine has a capacity given in GW_electric -> ```hasCapacityVariable = True```
- Emitting CO2 into the environment is not per se limited by a capacity -> ```hasCapacityVariable = False```

## Optional Parameters : Technical Parameters

When it comes to storage, it is useful to further specify its technical characteristics in more detail. The following parameters define the main technical specifications related to storage components.

### charge/dischargeRate & charge/dischargeEfficiency

`chargeRate` and `dischargeRate` define the ratio of the maximum storage inflow and outflow (in commodityUnit/hour) to the storage capacity (in commodityUnit), respectively. 

`chargeEfficiency` and `dischargeEfficiency` define the efficiency with which the storage can be charged. This corresponds to the percentage of the injected commodity that is transformed into stored commodity. 

Example: 

A battery system has a storage capacity of 100 MWh_elec.
It can be charged at a maximum rate of 20 MWh_elec per hour, meaning the `chargeRate` = 20 / 100 = 0.2 h⁻¹.

The `chargeEfficiency` is 0.9, meaning that only 90% of the electricity taken is effectively stored, while 10% is lost during the charging process.


### Charge/DischargeOpRateMax & Charge/DischargeOpRateFix

`ChargeOpRateMax` and `DischargeOpRateMax` define a maximum charging/discharging rate for each location and each time step, if required also for each investment period, by a positive float. It depends on `hasCapacityVariable` as follows:

- If ```hasCapacityVariable = True```, the values are given relative to the installed capacities (i.e. a value of 1 indicates a utilization of 100% of the capacity).
- If ```hasCapacityVariable = False```, the values are given as absolute values in form of the `commodityUnit` for each time step.

Type:

- None (default)
- Pandas DataFrame with positive (>= 0) entries. The row indices have to match the in the energy system model specified time steps. The column indices have to equal the in the energy system model specified locations. The data in ineligible locations are set to zero.
- Dictionary with investment periods as keys and one of the two options above as values


`Charge/DischargeOpRateFix` works the same way, by defining a fixed charging/discharging rate.

### capacityMax, CapacityMin and CapacityFix

`capacityMax` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#capacityMax)

### investPerCapacity

`investperCapacity` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#investPerCapacity)

### opexPerCapacity

`opexPerCapacity` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#opexPerCapacity)


### opexPerChargeOperation

`opexPerChargeOperation` describes the cost for one unit of the charge operation. The cost which is directly proportional to the charge operation of the component is obtained by multiplying the opexPerChargeOperation parameter with the annual sum of the operational time series of the components. The opexPerChargeOperation can either be given as a float or a Pandas Series with location specific values or a dictionary per investment period with one of the two previous options. The cost unit in which the parameter is given has to match the one specified in the energy system model (e.g. Euro, Dollar, 1e6 Euro). |br| * the default value is 0 :type opexPerChargeOperation: positive (>=0) float or Pandas Series with positive (>=0) values or dict of positive (>=0) float or Pandas Series with positive (>=0) values per investment period. The indices of the series have to equal the in the energy system model specified locations.

### interestRate

`interestRate` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#interestRate)

### economicLifetime

`economicLifetime` was already defined previously, but you can [see it here!](_2_add_sink.ipynb#economicLifetime)

Many parameters were left out here. Some of them might need a page on their own. Others could be collected in an "other features" notebook

## List of all parameters

Below, after executing the code cell, you will find the list of all parameters of a Storage Component, along with their description, type, and default value.

In [5]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "NetCDF"))
from docstringTable import display_param_table


display_param_table(fn.Storage)

,Description,Type,Default
Argument,,,
esM,energy system model to which the component should be added. Used for unit checks.,EnergySystemModel instance from the FINE package,/
name,name of the component. Has to be unique (i.e. no other components with that name can already exist in the EnergySystemModel instance to which the component is added).,string,/
commodity,to the component related commodity.,string,/
chargeRate,ratio of the maximum storage inflow (in commodityUnit/hour) to the storage capacity (in commodityUnit). Example: A hydrogen salt cavern which can store 133 GWh_H2_LHV can be charged 0.45 GWh_H2_LHV during one hour. The chargeRate thus equals 0.45/133 1/h.,0 < float,1
dischargeRate,ratio of the maximum storage outflow (in commodityUnit/hour) to the storage capacity (in commodityUnit). Example: A hydrogen salt cavern which can store 133 GWh_H2_LHV can be discharged 0.45 GWh_H2_LHV during one hour. The dischargeRate thus equals 0.45/133.,0 < float,1
chargeEfficiency,defines the efficiency with which the storage can be charged (equals the percentage of the injected commodity that is transformed into stored commodity). Enter 0.98 for 98% etc.,0 <= float <=1,1
dischargeEfficiency,defines the efficiency with which the storage can be discharged (equals the percentage of the withdrawn commodity that is transformed into stored commodity). Enter 0.98 for 98% etc.,0 <= float <=1,1
selfDischarge,percentage of self-discharge from the storage during one hour,0 <= float <=1,0
cyclicLifetime,"if specified, the total number of full cycle equivalents that are supported by the technology.",None or positive float,None
